In [46]:
import pandas as pd
import numpy as np

In [47]:
df = pd.read_csv('Dataset/cleaned_dataset.csv')
df.head(2)

,Employee_Name,EmpID,MarriedID,MaritalStatusID,GenderID,EmpStatusID,DeptID,PerfScoreID,FromDiversityJobFairID,Salary,...,ManagerName,ManagerID,RecruitmentSource,PerformanceScore,EngagementSurvey,EmpSatisfaction,SpecialProjectsCount,LastPerformanceReview_Date,DaysLateLast30,Absences
0,Adinolfi Wilson K,10026,0,0,1,1,5,4,0,62506,...,Michael Albert,22,LinkedIn,Exceeds,4.60,5,0,2019-01-17,0,1
1,Ait Sidi Karthikeyan,10084,1,1,1,5,3,3,0,104437,...,Simon Roup,4,Indeed,Fully Meets,4.96,3,6,2016-02-24,0,17


In [48]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 311 entries, 0 to 310
Data columns (total 36 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   Employee_Name               311 non-null    object 
 1   EmpID                       311 non-null    int64  
 2   MarriedID                   311 non-null    int64  
 3   MaritalStatusID             311 non-null    int64  
 4   GenderID                    311 non-null    int64  
 5   EmpStatusID                 311 non-null    int64  
 6   DeptID                      311 non-null    int64  
 7   PerfScoreID                 311 non-null    int64  
 8   FromDiversityJobFairID      311 non-null    int64  
 9   Salary                      311 non-null    int64  
 10  Termd                       311 non-null    int64  
 11  PositionID                  311 non-null    int64  
 12  Position                    311 non-null    object 
 13  State                       311 non

In [49]:
def convertDateTime(dataframe, *args):
    for col in args:
        dataframe[col] = pd.to_datetime(dataframe[col], format = 'mixed')

convertDateTime(df, 'DOB', 'DateofHire', 'DateofTermination', 'LastPerformanceReview_Date')

In [50]:
df['HireYear'] = df['DateofHire'].dt.year
df['HireMonth'] = df['DateofHire'].dt.month
df['HireDay'] = df['DateofHire'].dt.day
df.head(2)

,Employee_Name,EmpID,MarriedID,MaritalStatusID,GenderID,EmpStatusID,DeptID,PerfScoreID,FromDiversityJobFairID,Salary,...,PerformanceScore,EngagementSurvey,EmpSatisfaction,SpecialProjectsCount,LastPerformanceReview_Date,DaysLateLast30,Absences,HireYear,HireMonth,HireDay
0,Adinolfi Wilson K,10026,0,0,1,1,5,4,0,62506,...,Exceeds,4.60,5,0,2019-01-17,0,1,2011,7,5
1,Ait Sidi Karthikeyan,10084,1,1,1,5,3,3,0,104437,...,Fully Meets,4.96,3,6,2016-02-24,0,17,2015,3,30


In [51]:
df['HireMonthName'] = df['DateofHire'].dt.month_name()
df['HireDayName'] = df['DateofHire'].dt.day_name()
df[['HireMonthName', 'HireDayName']].head()

,HireMonthName,HireDayName
0,July,Tuesday
1,March,Monday
2,July,Tuesday
3,January,Monday
4,July,Monday


In [52]:
print(df['Salary'].max())
print(df['Salary'].min())

250000
45046


In [53]:
# salary - 45000 - 70000 -> low salary
# salary - 70001 - 150000 -> average salary
# salary - 150001 -> high salary

In [54]:
def salaryLabel(salary):
    if salary < 70000:
        return 'low salary'
    elif salary > 70000 and salary <= 150000:
        return "average salary"
    elif salary > 150000:
        return "high salary"
    else:
        return "Invalid salary !!!"

In [55]:
df['SalaryLabel'] = df['Salary'].apply(salaryLabel)

In [56]:
df[['Salary', 'SalaryLabel']].head()

,Salary,SalaryLabel
0,62506,low salary
1,104437,average salary
2,64955,low salary
3,64991,low salary
4,50825,low salary


In [57]:
df['SalaryLabel'].value_counts()

SalaryLabel
low salary        222
average salary     82
high salary         7
Name: count, dtype: int64

In [58]:

df['SalaryLabel'] = np.where(
    df['Salary'] < 70000,
    'low salary',
    np.where(
        df['Salary'] <= 150000,
        'average salary',
        'high salary'
    )
)

In [59]:

conditions = [
    df['Salary'] < 70000,
    df['Salary'] <= 150000,
    df['Salary'] > 150000
]

choices = [
    'low salary',
    'average salary',
    'high salary'
]

df['SalaryLabel'] = np.select(conditions, choices, default='Invalid salary')

In [60]:
df['SalaryLabel'].value_counts()

SalaryLabel
low salary        222
average salary     82
high salary         7
Name: count, dtype: int64

In [61]:
df['SalaryLabel'].unique()

array(['low salary', 'average salary', 'high salary'], dtype=object)

In [62]:
salary_dict = {
    'low salary' : 0, 
    'average salary' : 1, 
    'high salary' : 2
}

df['Salary_Num'] = df['SalaryLabel'].map(salary_dict)
df[['SalaryLabel', 'Salary_Num']].head()

,SalaryLabel,Salary_Num
0,low salary,0
1,average salary,1
2,low salary,0
3,low salary,0
4,low salary,0


# split

In [63]:
# split(find, n = index, expand = True/False(Default))
split_salary = df['SalaryLabel'].str.split(' ',n = 1, expand = True)
split_salary

,0,1
0,low,salary
1,average,salary
2,low,salary
3,low,salary
4,low,salary
...,...,...
306,low,salary
307,low,salary
308,high,salary
309,average,salary


In [64]:
df['Employee_Name'].head()

0      Adinolfi  Wilson  K
1    Ait Sidi  Karthikeyan
2        Akinkuolie  Sarah
3             Alagbe Trina
4          Anderson  Carol
Name: Employee_Name, dtype: object

In [65]:
split_name = df['Employee_Name'].str.split(" ", n = 2, expand = True)
split_name

,0,1,2
0,Adinolfi,,Wilson K
1,Ait,Sidi,Karthikeyan
2,Akinkuolie,,Sarah
3,Alagbe,Trina,None
4,Anderson,,Carol
...,...,...,...
306,Woodson,,Jason
307,Ybarra,,Catherine
308,Zamora,,Jennifer
309,Zhou,,Julia


In [68]:
hr = pd.read_csv('Dataset/HR_Dataset Refresh.csv')
hr.head()

,Employee_Name,EmpID,MarriedID,MaritalStatusID,GenderID,EmpStatusID,DeptID,PerfScoreID,FromDiversityJobFairID,Salary,...,ManagerName,ManagerID,RecruitmentSource,PerformanceScore,EngagementSurvey,EmpSatisfaction,SpecialProjectsCount,LastPerformanceReview_Date,DaysLateLast30,Absences
0,"Adinolfi, Wilson K",10026,0,0,1,1,5,4,0,62506,...,Michael Albert,22.0,LinkedIn,Exceeds,4.60,5,0,1/17/2019,0,1
1,"Ait Sidi, Karthikeyan",10084,1,1,1,5,3,3,0,104437,...,Simon Roup,4.0,Indeed,Fully Meets,4.96,3,6,2/24/2016,0,17
2,"Akinkuolie, Sarah",10196,1,1,0,5,5,3,0,64955,...,Kissy Sullivan,20.0,LinkedIn,Fully Meets,3.02,3,0,5/15/2012,0,3
3,"Alagbe,Trina",10088,1,1,0,1,5,3,0,64991,...,Elijiah Gray,16.0,Indeed,Fully Meets,4.84,5,0,1/3/2019,0,15
4,"Anderson, Carol",10069,0,2,0,5,5,3,0,50825,...,Webster Butler,39.0,Google Search,Fully Meets,5.00,4,0,2/1/2016,0,2


In [71]:
last_first = hr['Employee_Name'].str.split(',', n=2, expand=True)
last_first

,0,1
0,Adinolfi,Wilson K
1,Ait Sidi,Karthikeyan
2,Akinkuolie,Sarah
3,Alagbe,Trina
4,Anderson,Carol
...,...,...
577,Anderson,Carol
578,Anderson,Linda
579,Andreola,Colby
580,Athwal,Sam


In [72]:
first_middle = last_first[1].str.strip().str.split(' ', n=1, expand = True)
first_middle

,0,1
0,Wilson,K
1,Karthikeyan,None
2,Sarah,None
3,Trina,None
4,Carol,None
...,...,...
577,Carol,None
578,Linda,None
579,Colby,None
580,Sam,None


In [73]:
df['First_Name'] = first_middle[0]
df['Middle_Name'] = first_middle[1]
df['Last_Name'] = last_first[0]
df[['First_Name', 'Middle_Name', 'Last_Name']].head()

,First_Name,Middle_Name,Last_Name
0,Wilson,K,Adinolfi
1,Karthikeyan,None,Ait Sidi
2,Sarah,None,Akinkuolie
3,Trina,None,Alagbe
4,Carol,None,Anderson
